Scraping

In [1]:
!pip install google-play-scraper

# Mengimpor pustaka google_play_scraper untuk mengakses ulasan dan informasi aplikasi dari Google Play Store.
from google_play_scraper import app, reviews, Sort, reviews_all

import pandas as pd # Pandas untuk manipulasi dan analisis data
pd.options.mode.chained_assignment = None # Menonaktifkan peringatan chaining
import numpy as np # NumPy untuk komputasi numerik
seed = 0
np.random.seed(seed) # Mengatur seed untuk reproduktibilitas
import matplotlib.pyplot as plt # Matplotlib untuk visualisasi data
import seaborn as sns # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi
from sklearn.metrics import accuracy_score

import datetime as dt # Manipulasi data waktu dan tanggal
import re # Modul untuk bekerja dengan ekspresi reguler
import string # Berisi konstanta string, seperti tanda baca
from nltk.tokenize import word_tokenize # Tokenisasi teks
from nltk.corpus import stopwords # Daftar kata-kata berhenti dalam teks

!pip install sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory # Menghapus kata-kata berhenti dalam bahasa Indonesia

from wordcloud import WordCloud # Membuat visualisasi berbentuk awan kata (word cloud) dari teks

import nltk # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt')
nltk.download('punkt_tab') # Mengunduh dataset punkt_tab yang diperlukan
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SKYL15H\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\SKYL15H\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SKYL15H\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
# Mengimpor pustaka google_play_scraper untuk mengakses ulasan dan informasi aplikasi dari Google Play Store.
from google_play_scraper import app, reviews_all, Sort

# Mengambil ulasan dari aplikasi Mobile Legends: Bang Bang
# ID paket yang benar untuk Mobile Legends adalah 'com.mobile.legends'
scrapreview = reviews_all(
    'com.mobile.legends',      # ID aplikasi yang benar
    lang='id',                 # Bahasa ulasan
    country='id',              # Negara
    sort=Sort.MOST_RELEVANT,   # Urutan ulasan
    count=1000                 # Jumlah maksimum ulasan yang ingin diambil
)

In [3]:
# Menyimpan ulasan dalam file CSV
import csv

with open('ulasan_aplikasi_game_ML.csv', mode='w', newline='', encoding='utf-8') as file:
  writer = csv.writer(file)
  writer.writerow(['review']) # Menulis header kolom
  for review in scrapreview:
    writer.writerow([review['content']]) # Menulis kkonten ulasan ke dalam file CSV

Analysis Sentiment

In [4]:
df = pd.read_csv('ulasan_aplikasi_game_ML.csv')
df.head()

,review
0,"Terlalu fokus jualan skin, Sistem matchmaking ..."
1,"ga mengenakan banget, dan sering terjadi. Jari..."
2,"yo moonton, berikut adalah list keresahan saya..."
3,Lebih bagus main HoK atau League of Legends Wi...
4,matchmaking yang tidak seimbang antara tim sen...


In [5]:
app_reviews_df = pd.DataFrame(scrapreview)
app_reviews_df.shape
app_reviews_df.head()
app_reviews_df.to_csv('ulasan_aplikasi_game_ML.csv', index=False)

# Membuat DataFrame dari hasil scrapreview
app_reviews_df = pd.DataFrame(scrapreview)

# Menghitung jumlah baris dan kolom dalam DataFrame
jumlah_ulasan, jumlah_kolom = app_reviews_df.shape

In [6]:
app_reviews_df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,df414e8d-3a62-47eb-bfb5-ba353e22581b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Terlalu fokus jualan skin, Sistem matchmaking ...",2,1,2.1.70.11747,2026-06-03 00:40:18,"Halo Kak,\nKami mohon maaf atas pengalaman kur...",2026-06-03 10:32:00,2.1.70.11747
1,3b764b68-8854-430e-802a-65a928f28882,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"ga mengenakan banget, dan sering terjadi. Jari...",1,3,2.1.70.11747,2026-06-01 23:54:15,"Halo Kak,\nMaaf atas masalah jaringan yang Kak...",2026-06-02 10:42:29,2.1.70.11747
2,479db4cd-1349-41a2-809d-62e71d9fb6ed,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"yo moonton, berikut adalah list keresahan saya...",5,3,2.1.67.11733,2026-05-29 02:42:54,None,NaT,2.1.67.11733
3,74eeee33-85ef-4aef-b28d-5e6eb7ce6664,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Lebih bagus main HoK atau League of Legends Wi...,1,2,2.1.67.11733,2026-06-01 13:28:45,"Halo Kak,\nKami mohon maaf atas pengalaman kur...",2026-06-01 15:16:21,2.1.67.11733
4,cd450c53-1bbd-4db5-8ebd-d43f1e555eb7,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,matchmaking yang tidak seimbang antara tim sen...,3,0,2.1.67.11733,2026-06-01 10:51:10,"Halo Kak,\nkami berkomitmen untuk menciptakan ...",2026-06-01 11:23:10,2.1.67.11733


In [7]:
app_reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279000 entries, 0 to 278999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   reviewId              279000 non-null  object        
 1   userName              279000 non-null  object        
 2   userImage             279000 non-null  object        
 3   content               278999 non-null  object        
 4   score                 279000 non-null  int64         
 5   thumbsUpCount         279000 non-null  int64         
 6   reviewCreatedVersion  212678 non-null  object        
 7   at                    279000 non-null  datetime64[ns]
 8   replyContent          17851 non-null   object        
 9   repliedAt             17851 non-null   datetime64[ns]
 10  appVersion            212678 non-null  object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 23.4+ MB


In [8]:
# Membuat DataFrame baru (clean_df) dengan menghapus baris yang memiliki nilai yang hilang (NaN) dari app_reviews_df
clean_df = app_reviews_df.dropna()

In [9]:
# Menghapus baris duplikat dari DataFrame clean_df
clean_df = clean_df.drop_duplicates()

# Menghitung jumlah baris dan kolom dalam DataFrame clean_df setelah menghapus duplikat
jumlah_ulasan_setelah_ulasan_duplikat, jumlah_kolom_setelah_hapus_duplikat =clean_df.shape

In [10]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14811 entries, 0 to 278970
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              14811 non-null  object        
 1   userName              14811 non-null  object        
 2   userImage             14811 non-null  object        
 3   content               14811 non-null  object        
 4   score                 14811 non-null  int64         
 5   thumbsUpCount         14811 non-null  int64         
 6   reviewCreatedVersion  14811 non-null  object        
 7   at                    14811 non-null  datetime64[ns]
 8   replyContent          14811 non-null  object        
 9   repliedAt             14811 non-null  datetime64[ns]
 10  appVersion            14811 non-null  object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 1.4+ MB


In [11]:
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

def cleaningText(text):
 text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
 text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
 text = re.sub(r'RT[\s]', '', text) # menghapus RT
 text = re.sub(r"http\S+", '', text) # menghapus link
 text = re.sub(r'[0-9]+', '', text) # menghapus angka
 text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka

 text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
 text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
 text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
 return text

def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
 text = text.lower()
 return text

def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
 text = word_tokenize(text)
 return text

def filteringText(text): # Menghapus stopwords dalam teks
 listStopwords = set(stopwords.words('indonesian'))
 listStopwords1 = set(stopwords.words('english'))
 listStopwords.update(listStopwords1)
 listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
 filtered = []
 for txt in text:
   if txt not in listStopwords:
     filtered.append(txt)
 return filtered # Moved return statement outside the loop

def stemmingText(text): # Mengurangi kata ke bentuk dasarnya yang menghilangkan imbuhan awalan dan akhiran atau ke akar kata
 # Membuat objek stemmer
 factory = StemmerFactory()
 stemmer = factory.create_stemmer()

 # Memecah teks menjadi daftar kata
 words = text.split()

 # Menerapkan stemming pada setiap kata dalam daftar
 stemmed_words = [stemmer.stem(word) for word in words]

 # Menggabungkan kata-kata yang telah distem
 stemmed_text = ' '.join(stemmed_words)

 return stemmed_text

def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
   sentence = ' '.join(word for word in list_words)
   return sentence

In [12]:
slangwords = {"@": "di", "abis": "habis", "wtb": "beli", "masi": "masih", "wts": "jual", "wtt": "tukar", "bgt": "banget", "maks": "maksimal"}
def fix_slangwords(text):
  words = text.split()
  fixed_words = []

  for word in words:
    if word.lower() in slangwords:
      fixed_words.append(slangwords[word.lower()])
    else:
        fixed_words.append(word)

  # Perbaikan: Tambahkan spasi antar kata
  fixed_text = ' '.join(fixed_words)
  return fixed_text

In [13]:
# Membersihkan teks dan menyimpannya di kolom 'text_clean'
clean_df['text_clean'] = clean_df['content'].apply(cleaningText)

# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
clean_df['text_casefoldingText'] = clean_df['text_clean'].apply(casefoldingText)

# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
clean_df['text_slangwords'] = clean_df['text_casefoldingText'].apply(fix_slangwords)

# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'
clean_df['text_tokenizingText'] = clean_df['text_slangwords'].apply(tokenizingText)

# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
clean_df['text_stopword'] = clean_df['text_tokenizingText'].apply(filteringText)

# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_akhir'
clean_df['text_akhir'] = clean_df['text_stopword'].apply(toSentence)

In [14]:
clean_df

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,text_clean,text_casefoldingText,text_slangwords,text_tokenizingText,text_stopword,text_akhir
0,df414e8d-3a62-47eb-bfb5-ba353e22581b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Terlalu fokus jualan skin, Sistem matchmaking ...",2,1,2.1.70.11747,2026-06-03 00:40:18,"Halo Kak,\nKami mohon maaf atas pengalaman kur...",2026-06-03 10:32:00,2.1.70.11747,Terlalu fokus jualan skin Sistem matchmaking s...,terlalu fokus jualan skin sistem matchmaking s...,terlalu fokus jualan skin sistem matchmaking s...,"[terlalu, fokus, jualan, skin, sistem, matchma...","[fokus, jualan, skin, sistem, matchmaking, bur...",fokus jualan skin sistem matchmaking buruk sol...
1,3b764b68-8854-430e-802a-65a928f28882,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"ga mengenakan banget, dan sering terjadi. Jari...",1,3,2.1.70.11747,2026-06-01 23:54:15,"Halo Kak,\nMaaf atas masalah jaringan yang Kak...",2026-06-02 10:42:29,2.1.70.11747,ga mengenakan banget dan sering terjadi Jaring...,ga mengenakan banget dan sering terjadi jaring...,ga mengenakan banget dan sering terjadi jaring...,"[ga, mengenakan, banget, dan, sering, terjadi,...","[mengenakan, banget, jaringan, hp, bagus, bagu...",mengenakan banget jaringan hp bagus bagus aja ...
3,74eeee33-85ef-4aef-b28d-5e6eb7ce6664,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Lebih bagus main HoK atau League of Legends Wi...,1,2,2.1.67.11733,2026-06-01 13:28:45,"Halo Kak,\nKami mohon maaf atas pengalaman kur...",2026-06-01 15:16:21,2.1.67.11733,Lebih bagus main HoK atau League of Legends Wi...,lebih bagus main hok atau league of legends wi...,lebih bagus main hok atau league of legends wi...,"[lebih, bagus, main, hok, atau, league, of, le...","[bagus, main, hok, league, legends, wild, rift...",bagus main hok league legends wild rift game s...
4,cd450c53-1bbd-4db5-8ebd-d43f1e555eb7,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,matchmaking yang tidak seimbang antara tim sen...,3,0,2.1.67.11733,2026-06-01 10:51:10,"Halo Kak,\nkami berkomitmen untuk menciptakan ...",2026-06-01 11:23:10,2.1.67.11733,matchmaking yang tidak seimbang antara tim sen...,matchmaking yang tidak seimbang antara tim sen...,matchmaking yang tidak seimbang antara tim sen...,"[matchmaking, yang, tidak, seimbang, antara, t...","[matchmaking, seimbang, tim, tim, lawan, terka...",matchmaking seimbang tim tim lawan terkadang t...
6,219c0ab7-8ac7-4e18-8ef7-aaa3a5711df7,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Tambahan sedikit. Bintang gw udh kyk ingus nai...,1,9,2.1.70.11747,2026-05-24 20:31:28,"Halo Kak,\nKami mohon maaf atas pengalaman kur...",2026-05-22 11:05:53,2.1.70.11747,Tambahan sedikit Bintang gw udh kyk ingus naik...,tambahan sedikit bintang gw udh kyk ingus naik...,tambahan sedikit bintang gw udh kyk ingus naik...,"[tambahan, sedikit, bintang, gw, udh, kyk, ing...","[tambahan, bintang, gw, udh, kyk, ingus, turun...",tambahan bintang gw udh kyk ingus turun turunn...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278729,934d7305-4ae9-4e17-a7d4-2cf74b9a2938,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Masa main malah bug sinyal padahal jaringan ba...,1,2,1.8.47.9191,2024-01-17 19:19:44,"Dear Hero,\nMohon maaf atas ketidaknyamananya ...",2024-01-21 11:31:28,1.8.47.9191,Masa main malah bug sinyal padahal jaringan ba...,masa main malah bug sinyal padahal jaringan ba...,masa main malah bug sinyal padahal jaringan ba...,"[masa, main, malah, bug, sinyal, padahal, jari...","[main, bug, sinyal, jaringan, bagus, aneh]",main bug sinyal jaringan bagus aneh
278791,0354593c-7122-47ca-ac94-c1df7e11ae35,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Woy anjj ngasih tim yang bener, gw dari awal s...",1,0,1.7.44.8111,2023-02-02 19:33:08,"Dear Hero,\nKami bertekad untuk menciptakan li...

In [15]:
import csv
import requests
from io import StringIO

# Membaca data kamus kata-kata positif
lexicon_positive = {}
response_pos = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv')
if response_pos.status_code == 200:
    reader = csv.reader(StringIO(response_pos.text))
    for row in reader:
        if len(row) >= 2:
            lexicon_positive[row[0]] = int(row[1])
else:
    print("Gagal mengambil data lexicon positif")

# Membaca data kamus kata-kata negatif
lexicon_negative = {}
response_neg = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv')
if response_neg.status_code == 200:
    reader = csv.reader(StringIO(response_neg.text))
    for row in reader:
        if len(row) >= 2:
            lexicon_negative[row[0]] = int(row[1])
    print("Lexicon data successfully loaded.")
else:
    print("Gagal mengambil data lexicon negatif")

# Menampilkan jumlah kata unik untuk verifikasi
print(f"Positive words: {len(lexicon_positive)}")
print(f"Negative words: {len(lexicon_negative)}")

Lexicon data successfully loaded.
Positive words: 3609
Negative words: 6607


In [16]:
# Fungsi untuk menentukan polaritas sentimen dari ulasan

def sentiment_analysis_lexicon_indonesia(text):
  score = 0

  # Pastikan input adalah list kata (token)
  for word in text:
    if word in lexicon_positive:
      score += lexicon_positive[word]
    if word in lexicon_negative:
      score += lexicon_negative[word]

  # Tentukan polaritas berdasarkan skor total
  if score >= 0:
    polarity = 'positive'
  else:
    polarity = 'negative'

  return score, polarity

In [17]:
# Menerapkan fungsi dan memecah hasilnya ke dalam dua kolom baru
results = clean_df['text_stopword'].apply(sentiment_analysis_lexicon_indonesia)
results = list(zip(*results))

clean_df['polarity_score'] = results[0]
clean_df['polarity'] = results[1]

print(clean_df['polarity'].value_counts())

polarity
negative    10218
positive     4593
Name: count, dtype: int64


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Pastikan kolom text_akhir diisi ulang setelah perbaikan fix_slangwords di atas
clean_df['text_slangwords'] = clean_df['text_casefoldingText'].apply(fix_slangwords)
clean_df['text_tokenizingText'] = clean_df['text_slangwords'].apply(tokenizingText)
clean_df['text_stopword'] = clean_df['text_tokenizingText'].apply(filteringText)
clean_df['text_akhir'] = clean_df['text_stopword'].apply(toSentence)

X = clean_df['text_akhir']
y = clean_df['polarity']

# Ekstraksi fitur dengan TF-IDF (menurunkan min_df agar lebih fleksibel)
tfidf = TfidfVectorizer(max_features=200, min_df=5, max_df=0.8)
X_tfidf = tfidf.fit_transform(X)

features_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
display(features_df.head())

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

,adil,afk,aja,akun,ama,aneh,aplikasi,bagus,ban,banget,...,tpi,trus,tuh,turun,udah,udh,update,war,wifi,yg
0,0.0,0.0,0.00000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.271669
1,0.0,0.0,0.23058,0.000000,0.0,0.0,0.0,0.429105,0.0,0.267076,...,0.40939,0.0,0.0,0.000000,0.0,0.000000,0.0,0.385835,0.000000,0.000000
2,0.0,0.0,0.00000,0.211872,0.0,0.0,0.0,0.254738,0.0,0.000000,...,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
3,0.0,0.0,0.00000,0.000000,0.0,0.0,0.0,0.100013,0.0,0.000000,...,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.159324,0.000000
4,0.0,0.0,0.00000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,...,0.00000,0.0,0.0,0.155087,0.0,0.154466,0.0,0.000000,0.000000,0.508327


In [19]:
# RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier

# Membuat objek model Random Forest
random_forest = RandomForestClassifier()

# Melatih model Random Forest pada data pelatihan
random_forest.fit(X_train.toarray(), y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_rf = random_forest.predict(X_train.toarray())
y_pred_test_rf = random_forest.predict(X_test.toarray())

# Evaluasi akurasi model Random Forest
accuracy_train_rf = accuracy_score(y_pred_train_rf, y_train)
accuracy_test_rf = accuracy_score(y_pred_test_rf, y_test)

# Menampilkan akurasi
print('Random Forest - accuracy_train:', accuracy_train_rf)
print('Random Forest - accuracy_test:', accuracy_test_rf)

Random Forest - accuracy_train: 0.9944294395678596
Random Forest - accuracy_test: 0.80863989200135


EKSTRAKSI FITUR

In [20]:
!pip install gensim
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 2.8 MB/s eta 0:00:09
   -- ------------------------------------- 1.6/24.4 MB 4.0 MB/s eta 0:00:06
   --- ------------------------------------ 2.4/24.4 MB 4.1 MB/s eta 0:00:06
   ----- ---------------------------------- 3.1/24.4 MB 4.2 MB/s eta 0:00:06
   ------ --------------------------------- 4.2/24.4 MB 4.2 MB/s eta 0:00:05
   -------- ------------------------------- 5.0/24.4 MB 4.3 MB/s eta 0:00:05
   --------- ------------------------------ 5.8/24.4 MB 4.2 MB/s eta 0:00:05
   ---------- ----------------------------- 6.6/24.4 MB 4.2 MB/s eta 0:00:05
   ----------- ---------------------------- 7.1/24.4 MB 4.0 MB/s eta 0:00:05
   ------------- -------------------------- 8.1/24.4 MB 4.0 MB/s eta 0:00:05
   -------------- ------------------------- 8.9/24.4 MB 4.0 MB/s eta 0:00:04
   --------------- ------------------------ 9.7/24.4 MB 4.1 MB/s eta 0:00:04
   ---

In [21]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SKYL15H\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [22]:
text_data = [
 'Game paling ancur yang pernah ada',
 'Moonton tolong di baca',
 'Game nya tolong diperbaiki',
 'Aplikasi gak berguna',
 'Banyak Bug',
]

In [23]:
tokenized_data = [word_tokenize(sentence.lower()) for sentence in text_data]

In [24]:
model = Word2Vec(sentences=tokenized_data, vector_size=100, window=5, min_count=1, workers=4)

In [25]:
word_vectors = model.wv

# Menggunakan huruf kecil 'game' agar sesuai dengan data yang sudah di-lowercase sebelumnya
similar_words = word_vectors.most_similar('game', topn=3)
print("Kata-kata yang mirip dengan 'game':", similar_words)

vector = word_vectors['game']
print("Vektor untuk 'game':", vector)

Kata-kata yang mirip dengan 'game': [('yang', 0.16073349118232727), ('pernah', 0.1372736543416977), ('ancur', 0.1230054572224617)]
Vektor untuk 'game': [-8.6199632e-03  3.6655276e-03  5.1905052e-03  5.7412875e-03
  7.4670757e-03 -6.1673932e-03  1.1053649e-03  6.0477089e-03
 -2.8394996e-03 -6.1740419e-03 -4.1080380e-04 -8.3693014e-03
 -5.6007765e-03  7.1041510e-03  3.3523554e-03  7.2252541e-03
  6.8005561e-03  7.5310678e-03 -3.7887457e-03 -5.6226476e-04
  2.3481951e-03 -4.5187911e-03  8.3889691e-03 -9.8575167e-03
  6.7647509e-03  2.9152490e-03 -4.9337358e-03  4.3987250e-03
 -1.7397656e-03  6.7122946e-03  9.9643851e-03 -4.3621329e-03
 -5.9908372e-04 -5.6958455e-03  3.8505651e-03  2.7858252e-03
  6.8902457e-03  6.1007161e-03  9.5385769e-03  9.2734182e-03
  7.8979861e-03 -6.9891103e-03 -9.1559421e-03 -3.5569994e-04
 -3.1004576e-03  7.8933248e-03  5.9390636e-03 -1.5461347e-03
  1.5107561e-03  1.7903672e-03  7.8179538e-03 -9.5106894e-03
 -2.0551596e-04  3.4687072e-03 -9.3895267e-04  8.381639

KEKNYA OVERFITTING DEH

In [26]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier # Menggunakan Classifier karena ini tugas sentimen
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

# Menggunakan data dari clean_df yang sudah ada, bukan mengimpor dari sklearn.datasets
X = clean_df['text_akhir']
y = clean_df['polarity']

In [27]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# 1. Ekstraksi fitur ulang menggunakan Bigrams (1, 2) untuk menangkap konteks lebih baik
tfidf_optimized = TfidfVectorizer(ngram_range=(1, 2), max_features=2000, min_df=5)
X_tfidf_opt = tfidf_optimized.fit_transform(clean_df['text_akhir'])

# 2. Split data
X_train_opt, X_test_opt, y_train_opt, y_test_opt = train_test_split(
    X_tfidf_opt, clean_df['polarity'], test_size=0.2, random_state=42
)

# 3. Menggunakan LinearSVC (SVM) yang lebih stabil untuk teks
svm_model = LinearSVC(random_state=42, C=1.0)
svm_model.fit(X_train_opt, y_train_opt)

# 4. Evaluasi
y_pred_svm = svm_model.predict(X_test_opt)
print(f'Optimized SVM Accuracy: {accuracy_score(y_test_opt, y_pred_svm):.4f}')
print('\nClassification Report:')
print(classification_report(y_test_opt, y_pred_svm))

Optimized SVM Accuracy: 0.8984

Classification Report:
              precision    recall  f1-score   support

    negative       0.91      0.94      0.93      2039
    positive       0.86      0.81      0.83       924

    accuracy                           0.90      2963
   macro avg       0.89      0.87      0.88      2963
weighted avg       0.90      0.90      0.90      2963



NYOBA MAKE CROSS VALIDATION MWEHEHEHE

In [28]:
from sklearn.model_selection import cross_val_score

# Menjalankan 5-Fold Cross Validation menggunakan LinearSVC dan fitur TF-IDF yang dioptimasi
# X_tfidf_opt dan y_opt (polarity) diambil dari cell optimasi sebelumnya
cv_scores = cross_val_score(svm_model, X_tfidf_opt, clean_df['polarity'], cv=5)

print(f'Cross-Validation Scores: {cv_scores}')
print(f'Rata-rata Akurasi (Mean): {cv_scores.mean():.4f}')
print(f'Standar Deviasi: {cv_scores.std():.4f}')

Cross-Validation Scores: [0.88896389 0.89365294 0.8889264  0.88318704 0.88960162]
Rata-rata Akurasi (Mean): 0.8889
Standar Deviasi: 0.0033


In [29]:
pip freeze requirements.txt

absl-py==2.4.0
accelerate==1.13.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.5
aiosignal==1.4.0
altair==6.1.0
annotated-doc==0.0.4
annotated-types==0.7.0
anthropic==0.97.0
anyio==4.13.0
astor==0.8.1
asttokens==3.0.1
astunparse==1.6.3
async-timeout==4.0.3
attrs==26.1.0
backcall==0.2.0
bcrypt==5.0.0
beautifulsoup4==4.14.3
bitsandbytes==0.35.0
blake3==1.0.8
bleach==6.4.0
blinker==1.9.0
build==1.5.0
cachetools==7.1.0
cbor2==6.0.1
certifi==2026.4.22
cffi==2.0.0
charset-normalizer==3.4.7
chex==0.1.90
chromadb==1.5.9
click==8.3.3
cloudpickle==3.1.2
colorama==0.4.6
coloredlogs==15.0.1
comm==0.2.3
compressed-tensors==0.15.0.1
contourpy==1.3.2
cryptography==48.0.0
cycler==0.12.1
dataclasses-json==0.6.7
datasets==4.8.5
debugpy==1.8.20
decorator==5.2.1
deep-translator==1.11.4
defusedxml==0.7.1
depyf==0.20.0
diffusers==0.11.1
dill==0.4.1
diskcache==5.6.3
distro==1.9.0
dnspython==2.8.0
docopt==0.6.2
docstring_parser==0.18.0
durationpy==0.10
einops==0.8.2
email-validator==2.3.0
emoji==2.15.0
etils==1.13.0

In [30]:
# Instal pipreqs terlebih dahulu karena tidak tersedia secara default di Colab
!pip install pipreqs

# Jalankan pipreqs menggunakan tanda '!' dan pastikan path sesuai (menggunakan '.' untuk direktori saat ini)
!pipreqs . --force

INFO: Not scanning for jupyter notebooks.
INFO: Successfully saved requirements file in .\requirements.txt
